# **Regression: Gradient Boosting Regressor (GBR)**

## **Justification of Preprocessing Strategy**

### **Scale Invariance**
Like all tree-based ensembles, the Gradient Boosting Regressor is mathematically invariant to the scale of the input features. The underlying Decision Trees partition the data using discrete logical splits (e.g., `glucose > 120`) rather than calculating spatial distances. As a result, applying `StandardScaler` or `MinMaxScaler` would not change the tree structures or improve the loss optimization. To maximize computational efficiency and maintain clinical interpretability, we will train the model directly on the **Original, Unscaled Data**.

### **The Boosting Philosophy: Stochastic Weak Learners**
Gradient Boosting fits new trees to the residual errors of the previous trees. For this gradual learning process to work, the base estimators must be "Weak Learners" (typically trees with a `max_depth` of 3). If we were to inject our deep Decision Tree champion (`max_depth=20`) into this ensemble, the first tree would instantly memorize the training data, leaving a residual error of zero and completely breaking the boosting sequence. Therefore, we rely on the algorithm's native shallow trees, optimizing only the learning rate, ensemble size, and stochastic subsampling.

## **Experiment Design**

We structured a tournament of 3 optimization levels. Because sequential boosting can overfit if the learning rate is too high or the ensemble is too large, we rigorously log **both Train and Test metrics (RMSE, MAE, R²)** to monitor the generalization gap:

* **Baseline**: Scikit-Learn defaults (`n_estimators=100`, `learning_rate=0.1`, `max_depth=3`). This establishes our reference floor using standard weak learners.
* **GridSearchCV**: A targeted 3-fold cross-validated search exploring the classic boosting trade-off: ensemble size (`n_estimators`) versus step size (`learning_rate`), along with slight variations in tree depth.
* **Optuna Optimization**: Bayesian optimization deployed to fine-tune the continuous spaces of learning rate and tree depth. Crucially, we introduce `subsample` tuning to enable **Stochastic Gradient Boosting**, forcing each tree to train on a random fraction of the data to artificially increase diversity and combat overfitting.

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_GradientBoosting")

# 2. Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Apply One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target 
# Drop classification targets to prevent data leakage!
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage", "diabetes_risk_score"], axis=1)
y = df_final['diabetes_risk_score']

# Split data (80/20) - No stratify needed for continuous targets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

SEED = 42

def log_regression_metrics(y_tr_true, y_tr_pred, y_te_true, y_te_pred, duration):
    # Logs Train and Test metrics explicitly to monitor the Overfitting Gap
    # Train Partition Metrics
    mlflow.log_metric("rmse_train", mean_squared_error(y_tr_true, y_tr_pred) ** 0.5)
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr_true, y_tr_pred))
    mlflow.log_metric("r2_train", r2_score(y_tr_true, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("rmse_test", mean_squared_error(y_te_true, y_te_pred) ** 0.5)
    mlflow.log_metric("mae_test", mean_absolute_error(y_te_true, y_te_pred))
    mlflow.log_metric("r2_test", r2_score(y_te_true, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE
# ---------------------------------------------------------
with mlflow.start_run(run_name="GBR_Baseline"):
    reg_base = GradientBoostingRegressor(random_state=SEED)
    
    start_time = time.time()
    reg_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    # Explicit Predictions
    y_pred_train_base = reg_base.predict(X_train)
    y_pred_test_base = reg_base.predict(X_test)
    
    mlflow.log_params(reg_base.get_params())
    mlflow.log_param("optimization", "none_default")
    
    log_regression_metrics(y_train, y_pred_train_base, y_test, y_pred_test_base, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV
# ---------------------------------------------------------
with mlflow.start_run(run_name="GBR_GridSearch"):
    param_grid = {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5]
    }

    grid_reg = GridSearchCV(
        estimator=GradientBoostingRegressor(random_state=SEED),
        param_grid=param_grid,
        cv=KFold(n_splits=3, shuffle=True, random_state=SEED), # 3-fold for speed on 80k rows
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    start_time = time.time()
    grid_reg.fit(X_train, y_train)
    duration = time.time() - start_time

    best_gbr_grid = grid_reg.best_estimator_
    
    # Explicit Predictions
    y_pred_train_grid = best_gbr_grid.predict(X_train)
    y_pred_test_grid = best_gbr_grid.predict(X_test)

    mlflow.log_params(grid_reg.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_regression_metrics(y_train, y_pred_train_grid, y_test, y_pred_test_grid, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective_reg(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0) # Enables Stochastic Gradient Boosting
    }

    model = GradientBoostingRegressor(**params, random_state=SEED)
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=KFold(n_splits=3, shuffle=True, random_state=SEED), 
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    return -scores.mean()

with mlflow.start_run(run_name="GBR_Optuna"):
    study_reg = optuna.create_study(direction="minimize")
    
    start_time = time.time()
    study_reg.optimize(objective_reg, n_trials=12) # Safe trial budget
    duration = time.time() - start_time

    best_gbr_optuna = GradientBoostingRegressor(**study_reg.best_params, random_state=SEED)
    best_gbr_optuna.fit(X_train, y_train)
    
    # Explicit Predictions
    y_pred_train_optuna = best_gbr_optuna.predict(X_train)
    y_pred_test_optuna = best_gbr_optuna.predict(X_test)

    mlflow.log_params(study_reg.best_params)
    mlflow.log_param("optimization", "optuna")
    
    log_regression_metrics(y_train, y_pred_train_optuna, y_test, y_pred_test_optuna, duration)

2026/05/22 16:26:38 INFO mlflow.tracking.fluent: Experiment with name 'Regression_GradientBoosting' does not exist. Creating a new experiment.
[I 2026-05-22 16:31:53,587] A new study created in memory with name: no-name-268dbd00-5f5f-4077-bc07-8f2f648c0293
[I 2026-05-22 16:32:53,978] Trial 0 finished with value: 0.3279847417626265 and parameters: {'n_estimators': 187, 'learning_rate': 0.06304620877244686, 'max_depth': 6, 'subsample': 0.7295304949255506}. Best is trial 0 with value: 0.3279847417626265.
[I 2026-05-22 16:33:49,696] Trial 1 finished with value: 1.931783603695281 and parameters: {'n_estimators': 298, 'learning_rate': 0.012757782257170353, 'max_depth': 3, 'subsample': 0.8731291143402671}. Best is trial 0 with value: 0.3279847417626265.
[I 2026-05-22 16:34:28,006] Trial 2 finished with value: 0.36870334316838854 and parameters: {'n_estimators': 150, 'learning_rate': 0.10278338963729486, 'max_depth': 4, 'subsample': 0.9037985399262003}. Best is trial 0 with value: 0.3279847417

## Winner Run Selection (Priority Elimination Framework)

### Policy
A run is only eligible to win if it does NOT show evidence of overfitting or underfitting. Before applying the MAE/RMSE/R² decision rules, we require the Train→Test gaps to remain small enough to indicate acceptable generalization. Runs that memorize the training set or show a large Train/Test gap are disqualified regardless of metric rank.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** runs with overfitting or underfitting are removed from consideration.
2. **Priority 1 (60%): Lowest MAE (Test)** — primary objective for regression accuracy.
3. **Priority 2 (30%): Lowest RMSE (Test)** — used to reject runs where RMSE grows disproportionately relative to MAE.
4. **Priority 3 (10%): Acceptable R² (Test)** — confirms explanatory quality.
5. **Tiebreaker: Lowest Fit Time** — if MAE, RMSE, and R² are effectively tied.

### Runs Summary

| Run | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time |
|---|---:|---:|---:|---:|---:|---:|---:|
| GBR_Baseline | 0.43634 | 0.45093 | 0.56549 | 0.58591 | 0.99610 | 0.99584 | 30.38s |
| GBR_GridSearch | 0.22879 | 0.25192 | 0.28993 | 0.32201 | 0.99897 | 0.99874 | 283.63s |
| GBR_Optuna | 0.19643 | 0.22869 | 0.24923 | 0.29516 | 0.99924 | 0.99894 | 699.00s |

### Generalization Check (Test − Train)
- **GBR_Baseline:** MAE gap = 0.45093 − 0.43634 = **+0.01459** and RMSE gap = 0.58591 − 0.56549 = **+0.02042** → PASS.
- **GBR_GridSearch:** MAE gap = 0.25192 − 0.22879 = **+0.02314** and RMSE gap = 0.32201 − 0.28993 = **+0.03208** → PASS.
- **GBR_Optuna:** MAE gap = 0.22869 − 0.19643 = **+0.03226** and RMSE gap = 0.29516 − 0.24923 = **+0.04593** → PASS.

### Overfitting / Underfitting Validation
- None of the runs shows overfitting. The Train/Test gaps are small and stable across MAE and RMSE.
- None of the runs shows underfitting. All Test R² values are very high, so the model is capturing the target structure well.
- There is no evidence of catastrophic RMSE growth relative to MAE.

### Step-by-Step Elimination
**Step 1 — Apply the generalization filter**
- Passing runs: all three runs.

**Step 2 — Compare Test MAE (Priority 1 — 60%)**
- GBR_Optuna: 0.22869
- GBR_GridSearch: 0.25192
- GBR_Baseline: 0.45093
- Lowest MAE: **GBR_Optuna**.

**Step 3 — Compare Test RMSE (Priority 2 — 30%)**
- GBR_Optuna: 0.29516
- GBR_GridSearch: 0.32201
- GBR_Baseline: 0.58591
- GBR_Optuna remains the best choice.

**Step 4 — Check Test R² (Priority 3 — 10%)**
- GBR_Optuna: 0.99894
- GBR_GridSearch: 0.99874
- GBR_Baseline: 0.99584
- GBR_Optuna also leads on R².

### Final Decision
**Winner: GBR_Optuna**

**Justification:** `GBR_Optuna` is the strongest run among those that pass the generalization filter. It has the lowest Test MAE, the lowest Test RMSE, and the highest Test R². Fit time is not needed as a tiebreaker.

## Winner Hyperparameters
| Parameter | Value |
|---|---|
| **n_estimators** | 220 |
| **learning_rate** | 0.05490747097522296 |
| **max_depth** | 6 |
| **subsample** | 0.7038013432002436 |
| **random_state** | 42 |